# Spaceship Titanic - Baseline Model

This notebook builds a first end-to-end baseline model:
- Load `train.csv` and `test.csv`
- Apply cleaning/parsing features
- Build a preprocessing + model pipeline
- Evaluate with validation split and cross-validation
- Train on full data and export `submission.csv`


In [11]:
# Standard library path helper for robust file loading.
from pathlib import Path

# Core numerical/data libraries.
import numpy as np
import pandas as pd

# Modeling utilities from scikit-learn.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


In [12]:
# Resolve project root whether this notebook runs in notebooks/ or repo root.
base_dir = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

# Load Kaggle train/test data.
train = pd.read_csv(base_dir / 'train.csv')
test = pd.read_csv(base_dir / 'test.csv')

# Quick shape sanity check.
print('Train shape:', train.shape)
print('Test shape:', test.shape)


Train shape: (8693, 14)
Test shape: (4277, 13)


## Cleaning and Parsing Function

In [13]:
def clean_and_parse(df: pd.DataFrame) -> pd.DataFrame:
    # Work on a copy to avoid changing the input DataFrame in place.
    df = df.copy()

    # 1) Parse PassengerId (e.g., 0001_01) into group and member ids.
    pid = df['PassengerId'].astype(str).str.split('_', n=1, expand=True)
    df['GroupId'] = pid[0]
    df['GroupMemberId'] = pd.to_numeric(pid[1], errors='coerce')

    # 2) Parse Cabin (e.g., B/0/P) into deck, cabin number, and side.
    cabin = df['Cabin'].astype(str).str.split('/', n=2, expand=True)
    df['CabinDeck'] = cabin[0].replace('nan', np.nan)
    df['CabinNum'] = pd.to_numeric(cabin[1], errors='coerce')
    df['CabinSide'] = cabin[2].replace('nan', np.nan)

    # 3) Parse Name into first and last tokens.
    name = df['Name'].astype(str).str.split(' ', n=1, expand=True)
    df['FirstName'] = name[0].replace('nan', np.nan)
    df['LastName'] = name[1].replace('nan', np.nan)

    # 4) Build spend-based features.
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for c in spend_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    df['TotalSpend'] = df[spend_cols].fillna(0).sum(axis=1)
    df['NoSpend'] = (df['TotalSpend'] == 0).astype(int)

    # 5) Add group size feature.
    df['GroupSize'] = df.groupby('GroupId')['PassengerId'].transform('count')

    # 6) Normalize boolean-like columns as strings for sklearn stability across versions.
    # This avoids pandas nullable-boolean edge cases during imputation/encoding.
    for c in ['CryoSleep', 'VIP']:
        df[c] = df[c].map({True: 'True', False: 'False'})

    # Convert target dtype in training data only.
    if 'Transported' in df.columns:
        df['Transported'] = df['Transported'].astype(bool)

    return df


In [14]:
# Apply cleaning/parsing to both datasets.
train_clean = clean_and_parse(train)
test_clean = clean_and_parse(test)

# Check transformed shapes.
print('Train cleaned shape:', train_clean.shape)
print('Test cleaned shape:', test_clean.shape)


Train cleaned shape: (8693, 24)
Test cleaned shape: (4277, 23)


## Build Features and Target

In [15]:
# Define target and drop raw identifier/text columns.
target_col = 'Transported'
drop_cols = ['PassengerId', 'Name', 'Cabin']

# Build train features/labels and test features.
X = train_clean.drop(columns=[target_col]).drop(columns=drop_cols, errors='ignore')
y = train_clean[target_col].astype(int)
X_test = test_clean.drop(columns=drop_cols, errors='ignore')

# Convert bool-like columns to stable string categories before preprocessing.
for c in ['CryoSleep', 'VIP']:
    if c in X.columns:
        X[c] = X[c].astype(object)
    if c in X_test.columns:
        X_test[c] = X_test[c].astype(object)

print('X shape:', X.shape)
print('X_test shape:', X_test.shape)


X shape: (8693, 20)
X_test shape: (4277, 20)


In [16]:
# Split columns by dtype for tailored preprocessing.
cat_cols = X.select_dtypes(include=['object', 'category', 'boolean']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

# Numeric: median imputation.
# Categorical/boolean: mode imputation + one-hot encoding.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols),
    ]
)

# Baseline model.
model = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1)

# End-to-end pipeline: preprocessing then model.
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', model)
])


## Baseline Evaluation

In [17]:
# Holdout split for a fast local validation metric.
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline.fit(X_train, y_train)
val_preds = pipeline.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
print(f'Validation accuracy (single split): {val_acc:.4f}')

# Stratified 5-fold CV for a more stable estimate.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
print('CV accuracy scores:', np.round(cv_scores, 4))
print(f'CV mean accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')


Validation accuracy (single split): 0.7780
CV accuracy scores: [0.7993 0.7844 0.7901 0.7883 0.7785]
CV mean accuracy: 0.7881 +/- 0.0069


## Train on Full Data and Create Submission

In [18]:
# Fit on all available training data.
pipeline.fit(X, y)
test_preds = pipeline.predict(X_test).astype(bool)

# Build Kaggle submission file.
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': test_preds
})

# Save output to project root.
submission_path = base_dir / 'submission.csv'
submission.to_csv(submission_path, index=False)

print(f'Saved submission to: {submission_path}')
submission.head()


Saved submission to: /Users/salbo/Projects/Titanic_Spaceship/spaceship-titanic/submission.csv


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,False
